In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv


In [2]:
from sklearn.tree import DecisionTreeClassifier, export_graphviz
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.metrics import mean_squared_error

from sklearn.model_selection import train_test_split
from numpy import random

from fastai.tabular.all import *
from fastai.imports import *

In [3]:
df_train = pd.read_csv(r'/kaggle/input/playground-series-s6e2/train.csv')
df_test = pd.read_csv(r'/kaggle/input/playground-series-s6e2/test.csv')

df_submission = pd.read_csv(r'/kaggle/input/playground-series-s6e2/sample_submission.csv')

In [4]:
df_train.columns

Index(['id', 'Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
       'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
       'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium',
       'Heart Disease'],
      dtype='object')

In [5]:
splits = RandomSplitter(valid_pct=0.2, seed=42)(range_of(df_train))

In [6]:
to = TabularPandas(df_train, 
                   procs=[Categorify, FillMissing, Normalize],
                   cat_names = ['Sex', 'Chest pain type', 'EKG results', 'FBS over 120', 'Exercise angina', 'Thallium'],
                   cont_names = ['Age', 'BP', 'Cholesterol', 'Max HR', 'ST depression', 'Slope of ST', 'Number of vessels fluro'],
                   y_names='Heart Disease',
                   y_block=CategoryBlock(),
                   splits=splits)


In [7]:
# Get training and validation sets
X_train, y_train = to.train.xs, to.train.y
X_valid, y_valid = to.valid.xs, to.valid.y

In [8]:
# Initialize and train model
model_gpu = xgb.XGBClassifier(
    n_estimators=100,
    device="cuda", # Tells XGBoost to use NVIDIA GPU
    learning_rate=0.1,
    tree_method="auto" # Older version alternative
)
model_gpu.fit(X_train, y_train)


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [10:23:33] WARNING: /workspace/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [10:23:33] WARNING: /workspace/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

In [9]:
preds = model_gpu.predict(X_valid)

In [10]:
to.xs.iloc[:2]

,Sex,Chest pain type,EKG results,FBS over 120,Exercise angina,Thallium,Age,BP,Cholesterol,Max HR,ST depression,Slope of ST,Number of vessels fluro
371862,2,4,1,1,2,1,-1.590645,-0.166084,-0.327676,0.479467,0.721576,0.999037,0.686809
254771,2,3,3,1,1,3,-0.985051,-1.367939,-0.773013,-2.970915,1.143636,-0.835267,1.938326


In [12]:
dls = to.dataloaders(bs=64)

In [13]:
dls.show_batch()

,Sex,Chest pain type,EKG results,FBS over 120,Exercise angina,Thallium,Age,BP,Cholesterol,Max HR,ST depression,Slope of ST,Number of vessels fluro,Heart Disease
0,1,1,0,0,1,7,56.0,134.0,233.000000,141.999999,2.747608e-08,1.0,1.259037e-08,Presence
1,1,3,0,0,1,7,59.0,140.0,245.000000,125.000000,2.000000e+00,2.0,3.000000e+00,Presence
2,1,1,0,0,0,3,52.0,130.0,197.000002,146.000000,2.747608e-08,1.0,1.259037e-08,Absence
3,1,2,0,0,0,3,58.0,124.0,222.000000,155.000000,2.747608e-08,1.0,1.259037e-08,Absence
4,1,3,2,0,0,3,44.0,120.0,233.000000,169.000000,2.747608e-08,1.0,1.259037e-08,Absence
5,1,3,0,0,0,7,50.0,120.0,199.000001,174.999999,2.747608e-08,1.0,1.259037e-08,Absence
6,1,2,0,0,0,3,56.0,130.0,246.000000,162.000000,2.747608e-08,1.0,1.259037e-08,Absence
7,1,3,0,0,0,6,51.0,150.0,263.000000,170.000000,1.400000e+00,3.0,1.259037e-08,Absence
8,0,3,0,0,1,3,57.0,118.0,246.000000,141.999999,2.747608e-08,1.0,1.259037e-08,Absence
9,0,4,0,0,1,7,62.0,140.0,268.999999,168.000000,1.200000e+00,1.0,2.000000e+00,Presence


In [14]:
learn = tabular_learner(dls, metrics=accuracy)

In [15]:
dl_test = learn.dls.test_dl(df_test)

In [16]:
preds_test = model_gpu.predict(dl_test.xs)

In [17]:
preds_test.squeeze()[:5]

array([1, 0, 1, 0, 0])

In [18]:
df_submission.head()



,id,Heart Disease
0,630000,0
1,630001,0
2,630002,0
3,630003,0
4,630004,0


In [19]:
df_submission['heart_disease'] = preds_test
sub_df = df_submission[['id','heart_disease']]
sub_df.to_csv('submission.csv', index=False)

In [20]:


!head submission.csv



id,heart_disease
630000,1
630001,0
630002,1
630003,0
630004,0
630005,1
630006,0
630007,1
630008,1
